# Modul 7b: Skills bauen — Eigene Fähigkeiten entwickeln

**Agent Systems: Vom Konzept zum eigenen Orchestrator** | 🔴 Developer Track

In diesem Notebook baust du einen SKILL.md Parser, einen Shell-Script Generator und ein Test-Framework.

🎯 **Lernziel:** Einen vollständigen Skill (SKILL.md + Script) von Grund auf entwickeln und testen.

---

## Skill-Anatomie

Jeder Skill besteht aus:

```
skills/mein-skill/
├── SKILL.md        ← Frontmatter + Anweisungen
├── run.sh          ← Ausführbares Script
└── test.sh         ← Tests
```

**SKILL.md** hat YAML-Frontmatter für Metadaten und Markdown für Instructions:

```yaml
---
name: mein-skill
version: 1.0.0
triggers: [keyword1, keyword2]
author: jan
---
# Mein Skill
Anweisungen für den Agent...
```

In [ ]:
# SKILL.md Parser — liest Frontmatter und Instructions

def parse_skill_md(content):
    """Parst eine SKILL.md Datei in Frontmatter-Dict und Body."""
    lines = content.strip().split('\n')
    frontmatter = {}
    body_lines = []
    in_frontmatter = False
    frontmatter_done = False

    for line in lines:
        if line.strip() == '---' and not frontmatter_done:
            if not in_frontmatter:
                in_frontmatter = True
                continue
            else:
                in_frontmatter = False
                frontmatter_done = True
                continue

        if in_frontmatter:
            # Einfacher YAML-Parser (key: value)
            if ':' in line:
                key, val = line.split(':', 1)
                key = key.strip()
                val = val.strip()
                # Listen erkennen: [a, b, c]
                if val.startswith('[') and val.endswith(']'):
                    val = [v.strip().strip('"').strip("'") for v in val[1:-1].split(',')]
                frontmatter[key] = val
        elif frontmatter_done:
            body_lines.append(line)

    return frontmatter, '\n'.join(body_lines).strip()


# Test mit Beispiel-SKILL.md
sample_skill = '''---
name: deep-research
version: 1.0.0
triggers: [recherchiere, research, untersuche]
author: jan
category: research
approval: none
---
# Deep Research

Führe eine strukturierte Recherche durch:

1. Quellen sammeln (min. 3)
2. Zusammenfassen
3. Bewerten und Empfehlung geben

## Regeln
- Immer Quellen angeben
- Bei Unsicherheit kennzeichnen
'''

meta, body = parse_skill_md(sample_skill)
print('=== Frontmatter ===')
for k, v in meta.items():
    print(f'  {k}: {v}')
print(f'\n=== Instructions ({len(body)} Zeichen) ===')
print(body)

In [ ]:
# Shell-Script Generator und Test-Framework

def generate_run_script(meta, body):
    """Generiert ein run.sh Script aus Skill-Metadaten."""
    name = meta.get('name', 'unknown')
    lines = [
        '#!/bin/bash',
        f'# Auto-generated run.sh fuer Skill: {name}',
        f'# Version: {meta.get("version", "0.0.0")}',
        '',
        '# Argumente pruefen',
        'if [ -z "$1" ]; then',
        f'    echo "Usage: ./run.sh <input>"',
        '    exit 1',
        'fi',
        '',
        f'echo "[{name}] Starte mit Input: $1"',
        '',
        '# Hier kommt die Skill-Logik',
        f'echo "[{name}] Verarbeite..."',
        f'echo "[{name}] Fertig."',
    ]
    return '\n'.join(lines)


def generate_test_script(meta):
    """Generiert ein test.sh Script für den Skill."""
    name = meta.get('name', 'unknown')
    triggers = meta.get('triggers', [])
    lines = [
        '#!/bin/bash',
        f'# Auto-generated test.sh fuer Skill: {name}',
        '',
        'PASS=0',
        'FAIL=0',
        '',
        '# Test 1: Script existiert',
        'if [ -f "run.sh" ]; then',
        '    echo "PASS: run.sh existiert"',
        '    PASS=$((PASS + 1))',
        'else',
        '    echo "FAIL: run.sh fehlt"',
        '    FAIL=$((FAIL + 1))',
        'fi',
        '',
        '# Test 2: Script ist ausfuehrbar',
        'if [ -x "run.sh" ]; then',
        '    echo "PASS: run.sh ist ausfuehrbar"',
        '    PASS=$((PASS + 1))',
        'else',
        '    echo "FAIL: run.sh nicht ausfuehrbar"',
        '    FAIL=$((FAIL + 1))',
        'fi',
        '',
        '# Test 3: Ausfuehrung mit Test-Input',
        f'OUTPUT=$(./run.sh "test-input" 2>&1)',
        f'if echo "$OUTPUT" | grep -q "{name}"; then',
        '    echo "PASS: Output enthaelt Skill-Name"',
        '    PASS=$((PASS + 1))',
        'else',
        '    echo "FAIL: Output enthaelt nicht den Skill-Name"',
        '    FAIL=$((FAIL + 1))',
        'fi',
        '',
        'echo ""',
        'echo "Ergebnis: $PASS bestanden, $FAIL fehlgeschlagen"',
        '[ $FAIL -eq 0 ] && echo "ALL TESTS PASSED" || echo "SOME TESTS FAILED"',
    ]
    return '\n'.join(lines)


# Generieren und anzeigen
print('=== Generiertes run.sh ===')
run_sh = generate_run_script(meta, body)
print(run_sh)

print('\n=== Generiertes test.sh ===')
test_sh = generate_test_script(meta)
print(test_sh)

In [ ]:
# Python-basiertes Test-Framework (laeuft in Colab)

class SkillTestFramework:
    """Testet Skills ohne Shell — rein in Python."""
    def __init__(self):
        self.results = []

    def test(self, name, condition, message=''):
        passed = bool(condition)
        self.results.append({'name': name, 'passed': passed})
        icon = '✅' if passed else '❌'
        print(f'  {icon} {name} {("— " + message) if message else ""}')

    def summary(self):
        passed = sum(1 for r in self.results if r['passed'])
        total = len(self.results)
        print(f'\n📊 {passed}/{total} Tests bestanden')
        return passed == total


# Skill testen
print('=== Skill-Tests ===\n')
tf = SkillTestFramework()

# Test 1: Frontmatter hat alle Pflichtfelder
tf.test('Pflichtfelder vorhanden',
        all(k in meta for k in ['name', 'version', 'triggers']),
        f'Felder: {list(meta.keys())}')

# Test 2: Name ist gueltig (keine Leerzeichen)
tf.test('Name ist slug-format',
        ' ' not in meta.get('name', ''),
        f'Name: "{meta.get("name")}"')

# Test 3: Triggers sind eine Liste
tf.test('Triggers ist eine Liste',
        isinstance(meta.get('triggers'), list),
        f'Typ: {type(meta.get("triggers")).__name__}')

# Test 4: Mindestens 1 Trigger
tf.test('Mindestens 1 Trigger',
        len(meta.get('triggers', [])) >= 1,
        f'Anzahl: {len(meta.get("triggers", []))}')

# Test 5: Body ist nicht leer
tf.test('Instructions vorhanden',
        len(body) > 10,
        f'Laenge: {len(body)} Zeichen')

# Test 6: Version ist semantisch
version = meta.get('version', '')
tf.test('Version ist semver',
        len(version.split('.')) == 3,
        f'Version: {version}')

tf.summary()

In [ ]:
# 🎯 ÜBUNG: Schreibe einen eigenen Skill von Grund auf
#
# Aufgabe:
# 1. Definiere eine SKILL.md als String (mit Frontmatter + Instructions)
# 2. Parse sie mit parse_skill_md()
# 3. Generiere run.sh und test.sh
# 4. Teste mit dem SkillTestFramework

my_skill_md = '''---
name: 
triggers: []
version: 1.0.0
author: 
---
# TODO: Skill-Name

TODO: Instructions schreiben
'''

# TODO: Parsen
# my_meta, my_body = parse_skill_md(my_skill_md)

# TODO: Scripts generieren
# print(generate_run_script(my_meta, my_body))

# TODO: Testen
# tf2 = SkillTestFramework()
# tf2.test('Name vorhanden', my_meta.get('name') != '', ...)
# tf2.summary()

In [ ]:
# ✅ LÖSUNG

my_skill_md = '''---
name: daily-digest
triggers: [digest, zusammenfassung, tagesreview]
version: 1.0.0
author: jan
category: automation
approval: none
---
# Daily Digest

Erstelle eine tägliche Zusammenfassung:

1. Neue E-Mails zusammenfassen
2. Kalender-Termine auflisten
3. Offene TODOs zeigen
4. Wetter-Vorhersage anhängen

## Format
- Kurz und knapp (max 10 Zeilen)
- Emojis für Kategorien
- Prioritäten hervorheben
'''

my_meta, my_body = parse_skill_md(my_skill_md)
print('=== Mein Skill ===')
print(f'Name: {my_meta["name"]}')
print(f'Triggers: {my_meta["triggers"]}')
print(f'Instructions: {len(my_body)} Zeichen')

print('\n=== Generiertes run.sh ===')
print(generate_run_script(my_meta, my_body))

print('\n=== Tests ===')
tf2 = SkillTestFramework()
tf2.test('Pflichtfelder', all(k in my_meta for k in ['name', 'version', 'triggers']))
tf2.test('Slug-Format', ' ' not in my_meta['name'])
tf2.test('Triggers', isinstance(my_meta['triggers'], list) and len(my_meta['triggers']) >= 1)
tf2.test('Instructions', len(my_body) > 10)
tf2.test('Semver', len(my_meta['version'].split('.')) == 3)
tf2.summary()

## Skill-Development-Workflow

```
1. SKILL.md schreiben (Frontmatter + Instructions)
2. run.sh generieren / schreiben
3. test.sh ausführen → alle Tests grün
4. In skills/-Ordner ablegen
5. In AGENTS.md Routing-Tabelle eintragen
6. Optional: Auf ClawHub publizieren
```

---

### ✅ Self-Check
- [ ] Ich kann eine SKILL.md mit Frontmatter schreiben
- [ ] Ich verstehe den Zusammenhang: SKILL.md → run.sh → test.sh
- [ ] Ich kann einen Skill von Grund auf bauen und testen

**→ Weiter zu [Modul 8: Debugging & Ethik](modul8_debugging.ipynb)**